## 1. اتصال به پایگاه داده

تمام ارتباط‌ها از SQLAlchemy انجام می‌شود. اتصال‌ها با `with` بسته می‌شوند و مقادیر ورودی Queryها با bind parameter ارسال می‌شوند.


In [1]:
from pathlib import Path
from datetime import date
import os, re, statistics, subprocess, sys, time
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text, Integer, String, Numeric
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / "sql").exists() and (ROOT.parent / "sql").exists():
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

required = ["POSTGRES_DB", "POSTGRES_USER", "POSTGRES_PASSWORD"]
missing = [x for x in required if not os.getenv(x)]
if missing:
    raise RuntimeError(f"Missing .env values: {missing}")

url = URL.create(
    "postgresql+psycopg",
    username=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    host=os.getenv("DB_HOST") or "localhost",
    port=int(os.getenv("DB_PORT") or "5432"),
    database=os.getenv("POSTGRES_DB"),
)
engine = create_engine(url, pool_pre_ping=True)
reports_dir = ROOT / "reports"
print("Project:", ROOT)


Project: e:\project\FTI\Term3\ML&Python\02_Challenge_Database_Performance


In [2]:
with engine.connect() as conn:
    info = conn.execute(text("""
        SELECT current_database() AS database_name,
               current_user AS database_user,
               version() AS postgres_version
    """)).mappings().one()
display(pd.DataFrame([dict(info)]))

with engine.connect() as conn:
    objects = pd.read_sql_query(text("""
        SELECT table_schema, table_name, table_type
        FROM information_schema.tables
        WHERE table_schema IN ('olist', 'nyc311')
        ORDER BY table_schema, table_type, table_name
    """), conn)
display(objects)


,database_name,database_user,postgres_version
0,challenge_database,datafactory_user,PostgreSQL 16.15 (Debian 16.15-1.pgdg13+2) on ...


,table_schema,table_name,table_type
0,nyc311,cases,BASE TABLE
1,nyc311,cases_staging,BASE TABLE
2,olist,customers,BASE TABLE
3,olist,geolocation,BASE TABLE
4,olist,monthly_sales_cache,BASE TABLE
5,olist,order_items,BASE TABLE
6,olist,order_payments,BASE TABLE
7,olist,order_reviews,BASE TABLE
8,olist,orders,BASE TABLE
9,olist,product_category_translation,BASE TABLE


## 2. پنج سؤال تحلیلی و تعداد JOIN منطقی

| Query | جدول‌های اصلی | JOIN منطقی | عملیات غالب |
|---|---:|---:|---|
| Q1 — فروش ماهانه/دسته | ۴ | ۳ | Hash Join، Sort، GroupAggregate |
| Q2 — تأخیر/ایالت/ماه | ۲ | ۱ | Hash Join و تجمیع |
| Q3 — تأخیر/امتیاز نظر | ۲ | ۱ | Hash Join، محاسبه فاصله زمانی و تجمیع |
| Q4 — ۲۰ فروشنده برتر | ۴ | ۳ | چند Aggregate، Join و Sort |
| Q5 — روش پرداخت/ارزش سفارش | یک جدول فیزیکی با دو ارجاع | ۱ (CROSS JOIN) | تجمیع سفارش، CORR و تجمیع نهایی |

تعداد JOIN منطقی از SQL شمرده شده است. تعداد و نوع Join واقعیِ انتخاب‌شده توسط PostgreSQL در پلن‌ها استخراج می‌شود؛ این دو الزاماً یکسان نیستند. تعداد JOIN به‌تنهایی معیار سرعت نیست؛ حجم ردیف‌های میانی، Sort و Aggregate نیز مهم‌اند.


In [3]:
# بارگذاری Q1 تا Q5 از فایل SQL نسخه‌بندی‌شده
query_file = ROOT / "sql" / "queries" / "001_baseline_product_queries.sql"
source = query_file.read_text(encoding="utf-8")
pattern = re.compile(
    r"(?ms)^--\s*Q(?P<n>\d+):\s*(?P<title>[^\n]+)\n(?P<body>.*?)(?=^--\s*Q\d+:|\Z)"
)

queries = []
for match in pattern.finditer(source):
    raw = match.group("body").strip()
    sql = re.sub(r"^EXPLAIN\s*\([^)]*\)\s*", "", raw,
                 count=1, flags=re.I | re.S).strip().rstrip(";")
    queries.append({"n": int(match.group("n")), "title": match.group("title").strip(),
                    "sql": sql, "explain_sql": raw})

if len(queries) != 5:
    raise RuntimeError(f"Expected 5 query sections; found {len(queries)}. Check Q1-Q5 headings.")
print([(q["n"], q["title"]) for q in queries])


[(1, 'Monthly sales by product category'), (2, 'Delayed delivery rate by state and month'), (3, 'Relationship between delivery delay and review score'), (4, 'Top 20 sellers by revenue and performance'), (5, 'Payment distribution and relationship with order value')]


In [4]:
# اجرای پنج Query و نمایش نمونه خروجی
query_results = {}
for q in queries:
    with engine.connect() as conn:
        df = pd.read_sql_query(text(q["sql"]), conn)
    query_results[q["n"]] = df
    display(Markdown(f"### Q{q['n']} — {q['title']}"))
    print("Rows:", len(df))
    display(df.head(10))


### Q1 — Monthly sales by product category

Rows: 1282


,sales_month,category,total_sales,unique_orders,average_order_value
0,2016-09-01,furniture_decor,72.89,1,72.89
1,2016-09-01,health_beauty,134.97,1,134.97
2,2016-10-01,air_conditioning,1707.09,5,341.42
3,2016-10-01,audio,156.99,2,78.50
4,2016-10-01,auto,1408.25,9,156.47
5,2016-10-01,baby,1371.17,9,152.35
6,2016-10-01,bed_bath_table,478.99,5,95.80
7,2016-10-01,books_general_interest,119.50,1,119.50
8,2016-10-01,books_technical,267.00,1,267.00
9,2016-10-01,computers_accessories,1399.32,18,77.74


### Q2 — Delayed delivery rate by state and month

Rows: 556


,order_month,customer_state,delivered_orders,delayed_orders,delayed_percentage
0,2016-09-01,SP,1,1,100.0
1,2016-10-01,AL,1,0,0.0
2,2016-10-01,BA,3,0,0.0
3,2016-10-01,CE,6,0,0.0
4,2016-10-01,DF,6,0,0.0
5,2016-10-01,ES,3,0,0.0
6,2016-10-01,GO,7,0,0.0
7,2016-10-01,MA,4,1,25.0
8,2016-10-01,MG,35,0,0.0
9,2016-10-01,MT,1,0,0.0


... (5 outputs left out)

,seller_id,seller_state,seller_city,total_revenue,seller_orders,average_preparation_hours,average_review_score
0,4869f7a5dfa277a7dca6462dcf3b52b2,SP,guariba,229472.63,1132,56.17,4.13
1,53243585a1d6dc2643021fd1853d8905,BA,lauro de freitas,222776.05,358,73.22,4.13
2,4a3ca9315b744ce9f8e9374361493884,SP,ibitinga,200472.92,1806,56.34,3.83
3,fa1c13f2614d7b5c4749cbc52fecda94,SP,sumare,194042.03,585,56.09,4.34
4,7c67e1448b00f6e969d365cea6b010ab,SP,itaquaquecetuba,187923.89,982,273.22,3.49
5,7e93a43ef30c4f03f38b393420bc753a,SP,barueri,176431.87,336,57.10,4.21
6,da8622b14eb17ae2831f4ac5b9dab84a,SP,piracicaba,160236.57,1314,53.74,4.17
7,7a67c85e85bb2ce8582c35f2203ad736,SP,sao paulo,141745.53,1160,38.04,4.24
8,1025f0e2d44d7041d6cf58b6550e0bfa,SP,sao paulo,138968.55,915,89.77,3.99
9,955fee9216a65b617aa5c0531780ce60,SP,sao paulo,135171.70,1287,40.22,4.16


### Q5 — Payment distribution and relationship with order value

Rows: 28


,payment_type,payment_installments,payment_records,unique_orders,total_payment_value,average_payment_value,installment_order_value_correlation
0,boleto,1,19784,19784,2869361.27,145.03,0.3191
1,credit_card,0,2,2,188.63,94.32,0.3191
2,credit_card,1,25455,25407,2440445.43,95.87,0.3191
3,credit_card,2,12413,12389,1579283.03,127.23,0.3191
4,credit_card,3,10461,10443,1491103.80,142.54,0.3191
5,credit_card,4,7098,7088,1163907.61,163.98,0.3191
6,credit_card,5,5239,5234,961174.30,183.47,0.3191
7,credit_card,6,3920,3916,822611.81,209.85,0.3191
8,credit_card,7,1626,1623,305157.39,187.67,0.3191
9,credit_card,8,4268,4253,1313423.34,307.74,0.3191


In [5]:
# اندازه‌گیری تکرارپذیر سمت Python: یک warm-up و پنج اجرای واقعی، گزارش میانه
def benchmark_sql(sql, warmups=1, repetitions=5):
    values = []
    with engine.connect() as conn:
        for _ in range(warmups):
            conn.execute(text(sql)).fetchall()
        for _ in range(repetitions):
            started = time.perf_counter()
            rows = conn.execute(text(sql)).fetchall()
            values.append((time.perf_counter() - started) * 1000)
    return {"median_ms": statistics.median(values),
            "min_ms": min(values), "max_ms": max(values),
            "rows": len(rows), "runs_ms": [round(x, 2) for x in values]}

timing_rows = []
for q in queries:
    result = benchmark_sql(q["sql"])
    timing_rows.append({"query": f"Q{q['n']}", "title": q["title"],
                        **{k:v for k,v in result.items() if k != "runs_ms"}})
    print(f"Q{q['n']} runs (ms):", result["runs_ms"])
display(pd.DataFrame(timing_rows))


Q1 runs (ms): [352.99, 359.07, 382.5, 395.56, 393.55]
Q2 runs (ms): [153.16, 161.35, 158.38, 148.15, 144.82]
Q3 runs (ms): [160.87, 174.22, 168.71, 153.44, 147.34]
Q4 runs (ms): [871.55, 947.65, 898.43, 808.11, 871.79]
Q5 runs (ms): [462.94, 566.39, 414.07, 403.61, 384.39]


,query,title,median_ms,min_ms,max_ms,rows
0,Q1,Monthly sales by product category,382.5036,352.9850,395.5560,1282
1,Q2,Delayed delivery rate by state and month,153.1554,144.8191,161.3512,556
2,Q3,Relationship between delivery delay and review...,160.8737,147.3391,174.2247,20
3,Q4,Top 20 sellers by revenue and performance,871.7940,808.1089,947.6451,20
4,Q5,Payment distribution and relationship with ord...,414.0661,384.3931,566.3870,28


## 3. تحلیل Execution Plan و شمارش الگوریتم‌های Join

از گزارش‌های ذخیره‌شده در `reports/` زمان اجرا و تعداد nodeهای Join، Scan، Sort و Aggregate استخراج می‌شود.

- `Hash Join`: مناسب بسیاری از اتصال‌های حجیم بدون نیاز به ترتیب قبلی.
- `Merge Join`: وقتی ورودی‌ها مرتب باشند یا مرتب‌سازی به‌صرفه باشد.
- `Nested Loop`: مناسب ورودی کوچک یا دسترسی سریع به ورودی داخلی؛ روی دو ورودی بزرگ می‌تواند گران شود.
- `Seq Scan` الزاماً بد نیست؛ اگر بخش بزرگی از جدول لازم باشد، ممکن است ارزان‌تر از Index Scan باشد.


In [6]:
plan_files = {
    "baseline": "baseline_explain.txt",
    "optimized": "optimized_explain.txt",
    "candidate_optimizations": "candidate_optimizations_explain.txt",
    "q4_jit_off": "q4_jit_off_explain.txt",
    "q4_snapshot": "q4_snapshot_explain.txt",
    "q5_single_scan": "q5_single_scan_explain.txt",
    "q5_materialized_rows": "q5_materialized_rows_explain.txt",
    "q5_snapshot_final": "q5_snapshot_final_explain.txt",
    "workmem_64mb": "workmem_64mb_explain.txt",
}

def plan_blocks(content):
    blocks, current = [], []
    for line in content.splitlines():
        current.append(line)
        match = re.search(r"Execution Time:\s*([0-9.]+)\s*ms", line)
        if match:
            blocks.append(("\n".join(current), float(match.group(1))))
            current = []
    return blocks

def node_counts(block):
    patterns = {
        "hash_joins": r"^\s*Hash(?:\s+(?:Left|Right|Full))?\s+Join\b",
        "merge_joins": r"^\s*Merge(?:\s+(?:Left|Right|Full))?\s+Join\b",
        "nested_loops": r"^\s*Nested Loop(?:\s+(?:Left|Right|Full))?\b",
        "seq_scans": r"^\s*(?:Parallel )?Seq Scan\b",
        "index_scans": r"^\s*Index (?:Only )?Scan\b",
        "sorts": r"^\s*Sort\s+\(",
        "aggregates": r"^\s*(?:(?:Partial|Finalize) )?(?:HashAggregate|GroupAggregate)\b",
    }
    return {k: len(re.findall(v, block, flags=re.M)) for k,v in patterns.items()}

rows = []
for label, filename in plan_files.items():
    path = reports_dir / filename
    if not path.exists():
        continue
    content = path.read_text(encoding="utf-8", errors="replace")
    for i, (block, ms) in enumerate(plan_blocks(content), 1):
        counts = node_counts(block)
        rows.append({"report": label, "plan": i, "execution_ms": ms,
                     "total_join_nodes": counts["hash_joins"] + counts["merge_joins"] + counts["nested_loops"],
                     **counts})

plan_summary = pd.DataFrame(rows)
if plan_summary.empty:
    print("No EXPLAIN reports with Execution Time were found.")
else:
    display(plan_summary.sort_values(["report", "plan"]).reset_index(drop=True))


No EXPLAIN reports with Execution Time were found.


### تفسیر ساختاری پلن‌ها

- اختلاف `estimated rows` و `actual rows`: دقت تخمین‌زن PostgreSQL.
- `Sort Method: external merge`: Sort از حافظه فراتر رفته و از فایل موقت دیسک استفاده کرده است.
- `HashAggregate` و `GroupAggregate`: هزینه تجمیع روی ده‌ها/صدها هزار ردیف.
- تعداد ردیف‌های ورودی و خروجی هر Join: نشان می‌دهد کدام مرحله حجم داده میانی را بالا برده است.

برای Q4 و Q5، هزینه Aggregate و Sort را کنار Joinها تحلیل کن؛ صرفاً شمردن تعداد Join توضیح کافی نیست.


## 4. هزینه پنهان Indexها

گزارش اندازه Indexها و بنچمارک درج، مستقیماً از فایل‌های واقعی پروژه نمایش داده می‌شود.


In [7]:
def show_report(filename):
    path = reports_dir / filename
    display(Markdown(f"### `{filename}`"))
    if path.exists():
        print(path.read_text(encoding="utf-8", errors="replace"))
    else:
        print("Report not found:", path)

show_report("index_size_audit.txt")
show_report("index_write_benchmark.txt")


### `index_size_audit.txt`

��                      t a b l e _ n a m e                         |                           i n d e x _ n a m e                           |   i n d e x _ s i z e   |   t a b l e _ s i z e   |   i n d e x _ p c t   |   i n d e x _ s c a n s   |                                                                                                                                         i n d e x _ d e f i n i t i o n                                                                                                                                         
 
 - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - + - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - + - - - - - - - - - - - - + - - - - - - - - - - - - + - - - - - - - - - - - + - - - - - - - - - - - - - + - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 

### `index_write_benchmark.txt`

��T i m i n g   i s   o n . 
 
   c o u n t     
 
 - - - - - - - - 
 
   1 1 2 6 5 0 
 
 ( 1   r o w ) 
 
 
 
 T i m e :   1 2 3 . 7 9 5   m s 
 
   c o u n t   
 
 - - - - - - - 
 
   9 9 2 2 4 
 
 ( 1   r o w ) 
 
 
 
 T i m e :   7 5 . 5 5 2   m s 
 
   c o u n t     
 
 - - - - - - - - 
 
   1 0 3 8 8 6 
 
 ( 1   r o w ) 
 
 
 
 T i m e :   7 7 . 9 8 9   m s 
 
 C R E A T E   T A B L E   A S 
 
 T i m e :   4 9 . 8 9 5   m s 
 
 C R E A T E   T A B L E   A S 
 
 T i m e :   9 . 2 4 1   m s 
 
 C R E A T E   T A B L E   A S 
 
 T i m e :   5 . 4 1 0   m s 
 
 = = = = =   B A S E L I N E   I N S E R T   = = = = = 
 
                                                                                                                 Q U E R Y   P L A N                                                                                                                   
 
 - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 

### نتیجه بنچمارک INSERT ثبت‌شده

| جدول | بدون Index | با Index |
|---|---:|---:|
| `order_items` | 288.296 ms | 2085.290 ms |
| `order_reviews` | 452.180 ms | 880.666 ms |
| `order_payments` | 224.731 ms | 980.093 ms |

Indexها هزینه نگهداری دارند؛ در `order_items` زمان درج تقریباً ۷٫۲ برابر شده است. بنابراین Index باید با توجه به بهبود خواندن و هزینه نوشتن ارزیابی شود. Index حذف‌شده و دلیل حذف آن باید با نام دقیق و شاهد واقعی گزارش شود؛ نامی را حدس نزن.


## 5. View زنده در برابر Snapshot

برای تحلیل فروشنده‌ها، View زنده و جدول Snapshot با یک warm-up و هفت اجرای تکراری مقایسه می‌شوند. میانه زمان، حداقل/حداکثر و تعداد ردیف خروجی ثبت می‌شود.


In [8]:
def repeated_benchmark(sql, warmups=1, repetitions=7):
    times = []
    with engine.connect() as conn:
        for _ in range(warmups):
            conn.execute(text(sql)).fetchall()
        for _ in range(repetitions):
            start = time.perf_counter()
            result = conn.execute(text(sql)).fetchall()
            times.append((time.perf_counter() - start) * 1000)
    return {"median_ms": statistics.median(times), "min_ms": min(times),
            "max_ms": max(times), "rows": len(result),
            "runs_ms": [round(x, 2) for x in times]}

live_sql = """
SELECT seller_id, total_revenue
FROM olist.v_seller_performance_live
ORDER BY total_revenue DESC
LIMIT 20
"""
snapshot_sql = """
SELECT seller_id, total_revenue
FROM olist.seller_performance_snapshot
ORDER BY total_revenue DESC
LIMIT 20
"""

live_result = repeated_benchmark(live_sql)
snapshot_result = repeated_benchmark(snapshot_sql)

display(pd.DataFrame([
    {"source": "Live view", **{k:v for k,v in live_result.items() if k != "runs_ms"}},
    {"source": "Snapshot table", **{k:v for k,v in snapshot_result.items() if k != "runs_ms"}},
]))
print("Live view runs:", live_result["runs_ms"])
print("Snapshot runs:", snapshot_result["runs_ms"])


,source,median_ms,min_ms,max_ms,rows
0,Live view,142.1130,137.6608,193.0090,20
1,Snapshot table,1.8463,1.5515,2.9586,20


Live view runs: [193.01, 141.76, 142.11, 137.66, 138.58, 143.05, 146.15]
Snapshot runs: [2.46, 1.78, 1.64, 1.97, 2.96, 1.85, 1.55]


**تصمیم معماری:** Snapshot می‌تواند پاسخ سریع‌تری بدهد، اما تازگی داده را تا زمان Refresh از دست می‌دهد و فضای ذخیره‌سازی/نگهداری لازم دارد. View زنده داده تازه‌تر می‌دهد، اما هزینه محاسبه را هنگام هر خواندن می‌پردازد. انتخاب نهایی باید بر اساس اعداد همین بنچمارک باشد.


## 6. Function و Stored Procedure

- `olist.get_category_sales(start_date, end_date)` درون `SELECT` فراخوانی می‌شود و خروجی محاسبه‌شده برمی‌گرداند.
- `olist.refresh_monthly_sales(month)` با `CALL` اجرا می‌شود و Snapshot ماه را بازسازی می‌کند.
- اجرای دوباره Procedure برای یک ماه باید تعداد ردیف یکسانی تولید کند (idempotent).


In [9]:
# Function: پارامترها با bind parameters ارسال می‌شوند
with engine.connect() as conn:
    function_df = pd.read_sql_query(
        text("""
            SELECT *
            FROM olist.get_category_sales(
                CAST(:start_date AS date),
                CAST(:end_date AS date)
            )
            ORDER BY sales_month, category
        """),
        conn,
        params={"start_date": date(2017, 1, 1), "end_date": date(2017, 3, 31)}
    )
display(function_df.head(15))
print("Function rows:", len(function_df))

# Procedure: اجرای دوباره و بررسی idempotency
month = date(2017, 1, 1)
with engine.begin() as conn:
    conn.execute(text("CALL olist.refresh_monthly_sales(CAST(:month AS date))"), {"month": month})
    count_1 = conn.scalar(text("""
        SELECT COUNT(*) FROM olist.monthly_sales_cache WHERE sales_month = :month
    """), {"month": month})

    conn.execute(text("CALL olist.refresh_monthly_sales(CAST(:month AS date))"), {"month": month})
    count_2 = conn.scalar(text("""
        SELECT COUNT(*) FROM olist.monthly_sales_cache WHERE sales_month = :month
    """), {"month": month})

print({"first_run": count_1, "second_run": count_2, "idempotent": count_1 == count_2})


,sales_month,category,total_sales,unique_orders,average_order_value
0,2017-01-01,agro_industry_and_commerce,65.97,2,32.99
1,2017-01-01,air_conditioning,663.70,3,221.23
2,2017-01-01,auto,5168.83,29,178.24
3,2017-01-01,baby,6397.87,32,199.93
4,2017-01-01,bed_bath_table,3960.16,45,88.00
5,2017-01-01,books_general_interest,234.89,2,117.45
6,2017-01-01,computers,2199.99,1,2199.99
7,2017-01-01,computers_accessories,3924.14,24,163.51
8,2017-01-01,consoles_games,5954.20,21,283.53
9,2017-01-01,construction_tools_construction,310.00,1,310.00


Function rows: 150
{'first_run': 44, 'second_run': 44, 'idempotent': True}


## 7. پل Python و SQLAlchemy

این بخش خواندن chunk از View و نوشتن DataFrame با نگاشت صریح نوع ستون‌ها را نشان می‌دهد. تمام Connectionها با context manager بسته می‌شوند.


In [10]:
# خواندن View به‌صورت chunk
total_rows = 0
first_chunk = None
with engine.connect() as conn:
    for chunk in pd.read_sql_query(
        text("SELECT * FROM olist.v_delivered_orders"),
        conn,
        chunksize=10_000
    ):
        if first_chunk is None:
            first_chunk = chunk.copy()
        total_rows += len(chunk)

print("Rows read in chunks:", total_rows)
if first_chunk is not None:
    display(first_chunk.head())


Rows read in chunks: 96470


,order_id,customer_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26


In [11]:
# نوشتن DataFrame با نوع ستون‌های مشخص‌شده
demo_df = pd.DataFrame({
    "demo_id": pd.Series([1, 2, 3], dtype="int64"),
    "label": ["alpha", "beta", "gamma"],
    "amount": [10.25, 20.50, 30.75],
})

with engine.begin() as conn:
    demo_df.to_sql(
        "python_bridge_demo",
        con=conn,
        schema="olist",
        if_exists="replace",
        index=False,
        dtype={
            "demo_id": Integer(),
            "label": String(40),
            "amount": Numeric(12, 2),
        }
    )
    written = conn.scalar(text("SELECT COUNT(*) FROM olist.python_bridge_demo"))
    conn.execute(text("DROP TABLE olist.python_bridge_demo"))

print("Rows written with explicit SQL types:", written)


Rows written with explicit SQL types: 3


## 8. تست‌ها، Git و چک‌لیست تحویل

در اجرای قبلی، هر ۷ تست `test_database_objects.py` موفق بوده‌اند. سلول بعدی تست‌ها را دوباره اجرا می‌کند و تعداد Commitهای Git را گزارش می‌دهد.


In [12]:
test_run = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    cwd=str(ROOT), capture_output=True, text=True
)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
print("Test exit code:", test_run.returncode)
if test_run.returncode != 0:
    raise RuntimeError("Database tests failed.")

git_run = subprocess.run(
    ["git", "rev-list", "--count", "HEAD"],
    cwd=str(ROOT), capture_output=True, text=True
)
print("Git commits:", git_run.stdout.strip() if git_run.returncode == 0 else git_run.stderr)



test_category_sales_function_respects_date_range (test_database_objects.TestDatabaseObjects.test_category_sales_function_respects_date_range) ... ok
test_delivery_delay_query_on_controlled_data (test_database_objects.TestDatabaseObjects.test_delivery_delay_query_on_controlled_data) ... ok
test_monthly_procedure_is_idempotent (test_database_objects.TestDatabaseObjects.test_monthly_procedure_is_idempotent) ... ok
test_monthly_procedure_rejects_null_month (test_database_objects.TestDatabaseObjects.test_monthly_procedure_rejects_null_month) ... ok
test_payment_distribution_function_filters_type (test_database_objects.TestDatabaseObjects.test_payment_distribution_function_filters_type) ... ok
test_top_sellers_function_matches_snapshot (test_database_objects.TestDatabaseObjects.test_top_sellers_function_matches_snapshot) ... ok
test_views_match_source_data (test_database_objects.TestDatabaseObjects.test_views_match_source_data) ... ok

-------------------------------------------------------

## 9. چک‌لیست نهایی

- [ ] هر پنج Query و خروجی آن‌ها در Notebook اجرا شده باشد.
- [ ] پلن قبل و بعد برای هر Query موجود باشد.
- [ ] تعداد و نوع Joinهای واقعی، Scanها، Sortها و Aggregateها تحلیل شده باشد.
- [ ] برای هر Index توجیه مبتنی بر پلن وجود داشته باشد.
- [ ] اندازه Indexها و اثر آن‌ها بر INSERT گزارش شده باشد.
- [ ] دست‌کم یک Index حذف‌شده با نام و دلیل مستند شده باشد.
- [ ] View زنده و Snapshot با اندازه‌گیری تکرارشونده مقایسه شده باشند.
- [ ] Function و Procedure با پارامتر فراخوانی شده باشند.
- [ ] خواندن chunk، نوشتن DataFrame با نوع صریح و خواندن از View نشان داده شده باشد.
- [ ] حداقل ۶ تست موفق باشد.
- [ ] تمام DDLها در فایل‌های SQL نسخه‌بندی‌شده باشند.
- [ ] حداقل ۸ Commit معنادار در Git وجود داشته باشد.
- [ ] راه‌اندازی از PostgreSQL خالی با روال واحد و مستند انجام شود.

**بودجه ۲۰۰ ms:** Query بالاتر از ۲۰۰ ms را موفق اعلام نکن. زمان، پلن، گلوگاه، تغییرات امتحان‌شده و علت باقی‌ماندن هزینه را مستند کن. صورت مسئله برای Query ناموفق نیز توضیح فنی مستند می‌خواهد.


In [13]:
engine.dispose()
print("SQLAlchemy connection pool disposed.")


SQLAlchemy connection pool disposed.
